In [18]:
from datasets import load_dataset
import pandas as pd
import tqdm

### SQL QUERIES DATABASE

In [15]:
database = load_dataset("gretelai/synthetic_text_to_sql", split='train')
database = database.to_pandas()

In [46]:
database['sql_complexity']

0             single join
1             aggregation
2               basic SQL
3             aggregation
4        window functions
               ...       
99995           basic SQL
99996         single join
99997         single join
99998         single join
99999           basic SQL
Name: sql_complexity, Length: 100000, dtype: object

In [47]:
list(database[database['sql_complexity'] == 'single join']['sql'])[0]

'SELECT salesperson_id, name, SUM(volume) as total_volume FROM timber_sales JOIN salesperson ON timber_sales.salesperson_id = salesperson.salesperson_id GROUP BY salesperson_id, name ORDER BY total_volume DESC;'

In [36]:
df = database[['sql_prompt', 'sql']]
df

,sql_prompt,sql
0,What is the total volume of timber sold by eac...,"SELECT salesperson_id, name, SUM(volume) as to..."
1,List all the unique equipment types and their ...,"SELECT equipment_type, SUM(maintenance_frequen..."
2,How many marine species are found in the South...,SELECT COUNT(*) FROM marine_species WHERE loca...
3,What is the total trade value and average pric...,"SELECT trader_id, stock, SUM(price * quantity)..."
4,Find the energy efficiency upgrades with the h...,"SELECT type, cost FROM (SELECT type, cost, ROW..."
...,...,...
99995,Which programs had the highest volunteer parti...,"SELECT program_id, (num_volunteers / total_par..."
99996,What is the number of fair-trade certified acc...,SELECT COUNT(*) FROM products WHERE is_fair_tr...
99997,Find the user with the longest workout session...,"SELECT u.name, MAX(session_duration) as max_du..."
99998,How many space missions were completed by each...,"SELECT a.name, COUNT(sm.id) FROM Astronauts a ..."


In [18]:
df = df.tail(1000)
df.reset_index(drop=True, inplace=True)

In [19]:
df

,sql_prompt,sql
0,What is the average AI ethics budget per organ...,SELECT AVG(budget) OVER (PARTITION BY CASE WHE...
1,What is the total revenue for each music genre...,"SELECT Genre, SUM(Revenue) as Total_Revenue FR..."
2,Which marine protected areas have a maximum de...,SELECT area_name FROM shallow_protected_areas ...
3,What was the distribution of attendees by age ...,"SELECT event_id, age_group, COUNT(*) AS num_at..."
4,List all programs and their respective volunte...,"SELECT programs.name as program_name, managers..."
...,...,...
995,Which programs had the highest volunteer parti...,"SELECT program_id, (num_volunteers / total_par..."
996,What is the number of fair-trade certified acc...,SELECT COUNT(*) FROM products WHERE is_fair_tr...
997,Find the user with the longest workout session...,"SELECT u.name, MAX(session_duration) as max_du..."
998,How many space missions were completed by each...,"SELECT a.name, COUNT(sm.id) FROM Astronauts a ..."


In [13]:
# df.to_csv('data.csv', index=False)

### LIST BUCKETS AND RDS INSTANCES

In [2]:
from dotenv import load_dotenv
import os

load_dotenv()

True

In [3]:
import boto3

session = boto3.Session()

In [20]:
# List all S3 buckets

s3 = session.client('s3')
response = s3.list_buckets()
for bucket in response['Buckets']:
    print(f"Bucket Name: {bucket['Name']}")

Bucket Name: airflow-ml-models
Bucket Name: airflow-ml-orchestration
Bucket Name: codepipeline-eu-west-3-467013518129
Bucket Name: elasticbeanstalk-eu-west-3-128673362141
Bucket Name: elasticbeanstalk-us-west-2-128673362141
Bucket Name: pa-raw-data
Bucket Name: sk2-yasser-test
Bucket Name: zenml-train-data
Bucket Name: zenmlbucket


In [4]:
# List all RDS DB instances

rds = session.client('rds')
response = rds.describe_db_instances()
for db_instance in response['DBInstances']:
    print(f"DB Instance Identifier: {db_instance['DBInstanceIdentifier']}")

DB Instance Identifier: database-2


### DATABASE CONNEXION

In [7]:
# load_dotenv()

In [19]:
import pymysql

In [6]:
db = pymysql.connect(
        host = os.getenv('MYSQL_HOST'),
        user = os.getenv('MYSQL_USER'), 
        password = os.getenv('MYSQL_PASSWORD'),
        )

In [7]:
cursor = db.cursor()

In [8]:
cursor.execute('SHOW DATABASES')
cursor.fetchall()

(('information_schema',),
 ('mysql',),
 ('pa_sql_db',),
 ('performance_schema',),
 ('sys',))

In [12]:
# sql = "DROP DATABASE IF EXISTS SQL_PA;"

# # COMMIT TO SAVE CHANGES

# cursor.execute(sql)
# cursor.connection.commit()

In [9]:
# USE THE DATABASE 'pa_sql_db'

cursor.execute("USE pa_sql_db")

0

In [10]:
sql = "SHOW TABLES"
cursor.execute(sql)
cursor.fetchall()

()

### CREATE THE FOOTBALL TABLE

In [11]:
df = pd.read_csv('data.csv')
df.head(1)

,team,matches_played,wins,draws,losses,goal_difference,points
0,Real Madrid,486,291,85,110,533,533


In [12]:
sql_champions_league = '''
CREATE TABLE IF NOT EXISTS champions_league (
    team varchar(40),
    matches_played int NOT NULL, 
    wins int NOT NULL, 
    draws int NOT NULL, 
    losses int NOT NULL, 
    goal_difference int NOT NULL, 
    points int NOT NULL
    )
'''

In [13]:
try:
    cursor.execute('DROP TABLE IF EXISTS champions_league')
    cursor.execute(sql_champions_league)
    print('champions league table created !')

except Exception as e:
    print('ERROR : ', e)

champions league table created !


In [14]:
for i in tqdm.tqdm(range(len(df))):
    sql = "INSERT INTO champions_league (team, matches_played, wins, draws, losses, goal_difference, points) VALUES (%s, %s, %s, %s, %s, %s, %s)"
    cursor.execute(sql, tuple(df.iloc[i]))
    db.commit()

100%|██████████| 354/354 [00:04<00:00, 72.42it/s]


In [15]:
cursor.execute("SELECT * FROM champions_league")
cursor.fetchall()

(('Real Madrid', 486, 291, 85, 110, 533, 533),
 ('Bayern München', 388, 231, 78, 79, 427, 427),
 ('FC Barcelona', 341, 196, 77, 68, 319, 319),
 ('Manchester United', 289, 153, 69, 67, 224, 224),
 ('Juventus', 297, 151, 68, 78, 171, 171),
 ('AC Milan', 265, 127, 69, 69, 171, 171),
 ('Liverpool FC', 230, 128, 48, 54, 214, 214),
 ('FC Porto', 265, 120, 58, 87, 86, 86),
 ('SL Benfica', 266, 116, 61, 89, 114, 114),
 ('Chelsea FC', 197, 101, 52, 44, 153, 153),
 ('AFC Ajax', 215, 100, 50, 65, 105, 105),
 ('Inter', 203, 97, 53, 53, 77, 77),
 ('Arsenal FC', 197, 93, 44, 60, 98, 98),
 ('Borussia Dortmund', 176, 83, 36, 57, 75, 75),
 ('Atlético Madrid', 166, 79, 44, 43, 78, 78),
 ('Paris Saint-Germain', 151, 79, 29, 43, 115, 115),
 ('Manchester City', 127, 72, 26, 29, 123, 123),
 ('Dinamo Kiev', 186, 65, 41, 80, -22, -22),
 ('PSV Eindhoven', 171, 58, 43, 70, -11, -11),
 ('Celtic FC', 156, 61, 28, 67, -6, -6),
 ('Olympique Lyon', 136, 58, 36, 42, 49, 49),
 ('RSC Anderlecht', 163, 53, 35, 75, -64, 

In [16]:
cursor.connection.commit()

In [17]:
# show tables

cursor.execute("SHOW TABLES")
cursor.fetchall()

(('champions_league',),)

### CREATE THE FEEDBACK TABLE

In [5]:
from dotenv import load_dotenv
import pymysql
import os

load_dotenv()

True

In [6]:
db = pymysql.connect(
        host = os.getenv('FB_MYSQL_HOST'),
        database = os.getenv('FB_MYSQL_DATABASE'),
        user = os.getenv('FB_MYSQL_USER'), 
        password = os.getenv('FB_MYSQL_PASSWORD'),
        )

In [7]:
cursor = db.cursor()

In [8]:
cursor.execute('SHOW DATABASES')
cursor.fetchall()

(('feedback_db',),
 ('information_schema',),
 ('mysql',),
 ('performance_schema',),
 ('sys',))

In [9]:
cursor.execute("USE feedback_db")

0

In [10]:
sql = "SHOW TABLES"
cursor.execute(sql)
cursor.fetchall()

(('feedback_table',),)

In [26]:
sql_feedback = '''
CREATE TABLE IF NOT EXISTS feedback_table (
    sql_prompt varchar(400),
    sql_generated varchar(400),
    feedback tinyint NOT NULL
    )
'''

In [27]:
try:
    cursor.execute('DROP TABLE IF EXISTS feedback_table')
    cursor.execute(sql_feedback)
    print('feedback table created !')

except Exception as e:
    print('ERROR : ', e)

feedback table created !


In [31]:
sql = "SHOW TABLES"
cursor.execute(sql)
cursor.fetchall()

(('feedback_table',),)

In [11]:
cursor.execute("SELECT * FROM feedback_table")
cursor.fetchall()

(('Which team has won the most matches ?',
  'SELECT team FROM champions_league ORDER BY wins DESC LIMIT 1;',
  1),
 ('Which team has won the most matches ?',
  'SELECT team FROM champions_league ORDER BY wins DESC LIMIT 1;',
  1),
 ('Which team has won the most matches ? Give me the name and the number of wins',
  'SELECT team, wins FROM champions_league ORDER BY wins DESC LIMIT 1;',
  1),
 ('Which team has won the most matches ? Give me the name and the number of wins',
  'SELECT team, wins FROM champions_league ORDER BY wins DESC LIMIT 1;',
  1),
 ('Which team has won the most matches ? Give me the club name, the manager name and the number of wins',
  'SELECT t.team_name, t.manager, cl.wins FROM teams t JOIN champions_league cl ON t.team_name = cl.team ORDER BY cl.wins DESC LIMIT 1;',
  1),
 ('Which team has won the most matches ? Give me the club name, the manager name and the number of wins',
  'SELECT t.team_name, t.manager, cl.wins FROM champions_league cl INNER JOIN teams t ON

### ADD A SECOND TABLE IN PA_DB

In [18]:
from dotenv import load_dotenv
import pandas as pd
import pymysql
import tqdm
import os

load_dotenv()

True

In [ ]:
# MODIFY

# db = pymysql.connect(
#         host = os.getenv('MYSQL_HOST'),
#         database = os.getenv('MYSQL_DATABASE'),
#         user = os.getenv('MYSQL_USER'), 
#         password = os.getenv('MYSQL_PASSWORD'),
#         charset='utf8mb4'  # Ensure UTF-8 encoding
#         )

In [3]:
cursor = db.cursor()

In [4]:
sql = "SHOW TABLES"
cursor.execute(sql)
cursor.fetchall()

(('champions_league',), ('teams',))

In [28]:
sql_teams = '''
CREATE TABLE IF NOT EXISTS teams (
    team_name varchar(40),
    city varchar(40), 
    stadium varchar(40), 
    manager varchar(40)
    )
'''

In [29]:
try:
    cursor.execute('DROP TABLE IF EXISTS teams')
    cursor.execute(sql_teams)
    print('teams table created !')

except Exception as e:
    print('ERROR : ', e)

teams table created !


In [30]:
sql = "SHOW TABLES"
cursor.execute(sql)
cursor.fetchall()

(('champions_league',), ('teams',))

In [31]:
teams_data = [
    ('Real Madrid', 'Madrid', 'Santiago Bernabeu', 'Carlo Ancelotti'),
    ('Bayern Munchen', 'Munich', 'Allianz Arena', 'Julian Nagelsmann'),
    ('FC Barcelona', 'Barcelona', 'Camp Nou', 'Xavi Hernandez'),
    ('Manchester United', 'Manchester', 'Old Trafford', 'Erik ten Hag'),
    ('Juventus', 'Turin', 'Allianz Stadium', 'Massimiliano Allegri'),
    ('AC Milan', 'Milan', 'San Siro', 'Stefano Pioli'),
    ('Liverpool FC', 'Liverpool', 'Anfield', 'Jurgen Klopp'),
    ('FC Porto', 'Porto', 'Estadio do Dragao', 'Sergio Conceicao'),
    ('SL Benfica', 'Lisbon', 'Estadio da Luz', 'Roger Schmidt'),
    ('Chelsea FC', 'London', 'Stamford Bridge', 'Mauricio Pochettino'),
    ('AFC Ajax', 'Amsterdam', 'Johan Cruyff Arena', 'Maurice Steijn'),
    ('Inter', 'Milan', 'San Siro', 'Simone Inzaghi'),
    ('Arsenal FC', 'London', 'Emirates Stadium', 'Mikel Arteta'),
    ('Borussia Dortmund', 'Dortmund', 'Signal Iduna Park', 'Edin Terzic'),
    ('Atletico Madrid', 'Madrid', 'Wanda Metropolitano', 'Diego Simeone'),
    ('Paris Saint-Germain', 'Paris', 'Parc des Princes', 'Luis Enrique'),
    ('Manchester City', 'Manchester', 'Etihad Stadium', 'Pep Guardiola'),
    ('Dinamo Kiev', 'Kiev', 'NSC Olimpiyskiy', 'Mircea Lucescu'),
    ('PSV Eindhoven', 'Eindhoven', 'Philips Stadion', 'Peter Bosz'),
    ('Celtic FC', 'Glasgow', 'Celtic Park', 'Brendan Rodgers')
]

In [33]:
sql = "INSERT INTO teams (team_name, city, stadium, manager) VALUES (%s, %s, %s, %s)"

In [34]:
for team in teams_data:
    cursor.execute(sql, team)

In [35]:
cursor.connection.commit()

In [36]:
cursor.execute("SELECT * FROM teams")
cursor.fetchall()

(('Real Madrid', 'Madrid', 'Santiago Bernabeu', 'Carlo Ancelotti'),
 ('Bayern Munchen', 'Munich', 'Allianz Arena', 'Julian Nagelsmann'),
 ('FC Barcelona', 'Barcelona', 'Camp Nou', 'Xavi Hernandez'),
 ('Manchester United', 'Manchester', 'Old Trafford', 'Erik ten Hag'),
 ('Juventus', 'Turin', 'Allianz Stadium', 'Massimiliano Allegri'),
 ('AC Milan', 'Milan', 'San Siro', 'Stefano Pioli'),
 ('Liverpool FC', 'Liverpool', 'Anfield', 'Jurgen Klopp'),
 ('FC Porto', 'Porto', 'Estadio do Dragao', 'Sergio Conceicao'),
 ('SL Benfica', 'Lisbon', 'Estadio da Luz', 'Roger Schmidt'),
 ('Chelsea FC', 'London', 'Stamford Bridge', 'Mauricio Pochettino'),
 ('AFC Ajax', 'Amsterdam', 'Johan Cruyff Arena', 'Maurice Steijn'),
 ('Inter', 'Milan', 'San Siro', 'Simone Inzaghi'),
 ('Arsenal FC', 'London', 'Emirates Stadium', 'Mikel Arteta'),
 ('Borussia Dortmund', 'Dortmund', 'Signal Iduna Park', 'Edin Terzic'),
 ('Atletico Madrid', 'Madrid', 'Wanda Metropolitano', 'Diego Simeone'),
 ('Paris Saint-Germain', 'Pari

In [37]:
sql = "SHOW TABLES"
cursor.execute(sql)
cursor.fetchall()

(('champions_league',), ('teams',))

In [38]:
cursor.close()
db.close()